# Sentiment Analysis

## Import Library

In [1]:
import pandas as pd

import nltk
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
nltk.download('punkt')
nltk.download('stopwords')
import re
import emoji

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.svm import LinearSVC
from sklearn.metrics import classification_report, confusion_matrix

[nltk_data] Downloading package punkt to /home/fuaadd/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to /home/fuaadd/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


## Loading Dataset

In [2]:
reviews = pd.read_csv('./gojek_reviews.csv')
reviews.head()

,content,score,thumbsUpCount,reviewCreatedVersion,replyContent,appVersion
0,Aplikasinya sudah bagus meski terkadang ada bu...,4,37,5.36.2,NaN,5.36.2
1,Menu bantuan selalu dijawab oleh chatbot yang ...,1,8,5.36.2,"Hai Kak @supermeind, mohon maaf atas ketidakny...",5.36.2
2,"Fitur gofood nya kyk sampah, dikasih opsi peng...",1,108,5.32.1,"Hai Kak Rizi, mohon maaf atas ketidaknyamanann...",5.32.1
3,Tolong beri perlindungan untuk nasabah yang pe...,1,53,5.36.2,"Hai Kak Putu, mohon maaf atas ketidaknyamanann...",5.36.2
4,Pemakaian aplikasi mudah dan praktis. Promo ya...,5,81,5.37.1,NaN,5.37.1


In [3]:
reviews.info()
reviews.duplicated().sum()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20000 entries, 0 to 19999
Data columns (total 6 columns):
 #   Column                Non-Null Count  Dtype 
---  ------                --------------  ----- 
 0   content               20000 non-null  object
 1   score                 20000 non-null  int64 
 2   thumbsUpCount         20000 non-null  int64 
 3   reviewCreatedVersion  16174 non-null  object
 4   replyContent          9546 non-null   object
 5   appVersion            16174 non-null  object
dtypes: int64(2), object(4)
memory usage: 937.6+ KB


np.int64(6)

In [4]:
reviews.dropna(inplace=True)
reviews.drop_duplicates(inplace=True)
reviews.info()

<class 'pandas.core.frame.DataFrame'>
Index: 7628 entries, 1 to 19999
Data columns (total 6 columns):
 #   Column                Non-Null Count  Dtype 
---  ------                --------------  ----- 
 0   content               7628 non-null   object
 1   score                 7628 non-null   int64 
 2   thumbsUpCount         7628 non-null   int64 
 3   reviewCreatedVersion  7628 non-null   object
 4   replyContent          7628 non-null   object
 5   appVersion            7628 non-null   object
dtypes: int64(2), object(4)
memory usage: 417.2+ KB


## Text Preprocessing

In [5]:
# mengubah semua teks menjadi huruf kecil
reviews['lowercase_content'] = reviews['content'].map(lambda x: x.lower())
reviews.head()

,content,score,thumbsUpCount,reviewCreatedVersion,replyContent,appVersion,lowercase_content
1,Menu bantuan selalu dijawab oleh chatbot yang ...,1,8,5.36.2,"Hai Kak @supermeind, mohon maaf atas ketidakny...",5.36.2,menu bantuan selalu dijawab oleh chatbot yang ...
2,"Fitur gofood nya kyk sampah, dikasih opsi peng...",1,108,5.32.1,"Hai Kak Rizi, mohon maaf atas ketidaknyamanann...",5.32.1,"fitur gofood nya kyk sampah, dikasih opsi peng..."
3,Tolong beri perlindungan untuk nasabah yang pe...,1,53,5.36.2,"Hai Kak Putu, mohon maaf atas ketidaknyamanann...",5.36.2,tolong beri perlindungan untuk nasabah yang pe...
5,tolong dong di optimalkan lagi. awal awal saya...,3,3,5.37.1,"Hai Kak @Mr. ZimmermAn, mohon maaf atas ketida...",5.37.1,tolong dong di optimalkan lagi. awal awal saya...
6,TIDAK SOLUTIF sya order gofood dan pembayaran ...,1,6,5.36.2,"Hai Kak Rossy, mohon maaf atas ketidaknyamanan...",5.36.2,tidak solutif sya order gofood dan pembayaran ...


In [6]:
# Text cleaning
def clean_text(text):
    # menghapus emoji
    text = emoji.replace_emoji(text, replace='')
    # menghapus URL
    text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE)
    # menghapus karakter khusus dan angka
    text = re.sub(r'[^a-zA-Z\s]', ' ', text)
    # menghapus spasi berlebih
    text = re.sub(r'\s+', ' ', text).strip()
    
    return text
reviews['cleaned_content'] = reviews['lowercase_content'].apply(clean_text)
reviews.head()

,content,score,thumbsUpCount,reviewCreatedVersion,replyContent,appVersion,lowercase_content,cleaned_content
1,Menu bantuan selalu dijawab oleh chatbot yang ...,1,8,5.36.2,"Hai Kak @supermeind, mohon maaf atas ketidakny...",5.36.2,menu bantuan selalu dijawab oleh chatbot yang ...,menu bantuan selalu dijawab oleh chatbot yang ...
2,"Fitur gofood nya kyk sampah, dikasih opsi peng...",1,108,5.32.1,"Hai Kak Rizi, mohon maaf atas ketidaknyamanann...",5.32.1,"fitur gofood nya kyk sampah, dikasih opsi peng...",fitur gofood nya kyk sampah dikasih opsi penga...
3,Tolong beri perlindungan untuk nasabah yang pe...,1,53,5.36.2,"Hai Kak Putu, mohon maaf atas ketidaknyamanann...",5.36.2,tolong beri perlindungan untuk nasabah yang pe...,tolong beri perlindungan untuk nasabah yang pe...
5,tolong dong di optimalkan lagi. awal awal saya...,3,3,5.37.1,"Hai Kak @Mr. ZimmermAn, mohon maaf atas ketida...",5.37.1,tolong dong di optimalkan lagi. awal awal saya...,tolong dong di optimalkan lagi awal awal saya ...
6,TIDAK SOLUTIF sya order gofood dan pembayaran ...,1,6,5.36.2,"Hai Kak Rossy, mohon maaf atas ketidaknyamanan...",5.36.2,tidak solutif sya order gofood dan pembayaran ...,tidak solutif sya order gofood dan pembayaran ...


In [7]:
reviews["tokenized_content"] = reviews["cleaned_content"].apply(word_tokenize)
reviews.head()

,content,score,thumbsUpCount,reviewCreatedVersion,replyContent,appVersion,lowercase_content,cleaned_content,tokenized_content
1,Menu bantuan selalu dijawab oleh chatbot yang ...,1,8,5.36.2,"Hai Kak @supermeind, mohon maaf atas ketidakny...",5.36.2,menu bantuan selalu dijawab oleh chatbot yang ...,menu bantuan selalu dijawab oleh chatbot yang ...,"[menu, bantuan, selalu, dijawab, oleh, chatbot..."
2,"Fitur gofood nya kyk sampah, dikasih opsi peng...",1,108,5.32.1,"Hai Kak Rizi, mohon maaf atas ketidaknyamanann...",5.32.1,"fitur gofood nya kyk sampah, dikasih opsi peng...",fitur gofood nya kyk sampah dikasih opsi penga...,"[fitur, gofood, nya, kyk, sampah, dikasih, ops..."
3,Tolong beri perlindungan untuk nasabah yang pe...,1,53,5.36.2,"Hai Kak Putu, mohon maaf atas ketidaknyamanann...",5.36.2,tolong beri perlindungan untuk nasabah yang pe...,tolong beri perlindungan untuk nasabah yang pe...,"[tolong, beri, perlindungan, untuk, nasabah, y..."
5,tolong dong di optimalkan lagi. awal awal saya...,3,3,5.37.1,"Hai Kak @Mr. ZimmermAn, mohon maaf atas ketida...",5.37.1,tolong dong di optimalkan lagi. awal awal saya...,tolong dong di optimalkan lagi awal awal saya ...,"[tolong, dong, di, optimalkan, lagi, awal, awa..."
6,TIDAK SOLUTIF sya order gofood dan pembayaran ...,1,6,5.36.2,"Hai Kak Rossy, mohon maaf atas ketidaknyamanan...",5.36.2,tidak solutif sya order gofood dan pembayaran ...,tidak solutif sya order gofood dan pembayaran ...,"[tidak, solutif, sya, order, gofood, dan, pemb..."


In [8]:
# Normalisasi slangword
# kamus slangword diunduh dari project github : https://github.com/ramaprakoso/analisis-sentimen/blob/master/kamus/kbba.txt
slangword_dict = pd.read_table('slangwords.txt', header=None, sep='\t')
dico_slang = dict(zip(slangword_dict[0], slangword_dict[1]))
def normalize_slangwords(tokens, slang_dict):
    normalized_tokens = [slang_dict.get(token, token) for token in tokens]
    return normalized_tokens
reviews['normalized_content'] = reviews['tokenized_content'].apply(lambda x: normalize_slangwords(x, dico_slang))
reviews.head()

,content,score,thumbsUpCount,reviewCreatedVersion,replyContent,appVersion,lowercase_content,cleaned_content,tokenized_content,normalized_content
1,Menu bantuan selalu dijawab oleh chatbot yang ...,1,8,5.36.2,"Hai Kak @supermeind, mohon maaf atas ketidakny...",5.36.2,menu bantuan selalu dijawab oleh chatbot yang ...,menu bantuan selalu dijawab oleh chatbot yang ...,"[menu, bantuan, selalu, dijawab, oleh, chatbot...","[menu, bantuan, selalu, dijawab, oleh, chatbot..."
2,"Fitur gofood nya kyk sampah, dikasih opsi peng...",1,108,5.32.1,"Hai Kak Rizi, mohon maaf atas ketidaknyamanann...",5.32.1,"fitur gofood nya kyk sampah, dikasih opsi peng...",fitur gofood nya kyk sampah dikasih opsi penga...,"[fitur, gofood, nya, kyk, sampah, dikasih, ops...","[fitur, gofood, nya, seperti, sampah, dikasih,..."
3,Tolong beri perlindungan untuk nasabah yang pe...,1,53,5.36.2,"Hai Kak Putu, mohon maaf atas ketidaknyamanann...",5.36.2,tolong beri perlindungan untuk nasabah yang pe...,tolong beri perlindungan untuk nasabah yang pe...,"[tolong, beri, perlindungan, untuk, nasabah, y...","[tolong, beri, perlindungan, untuk, nasabah, y..."
5,tolong dong di optimalkan lagi. awal awal saya...,3,3,5.37.1,"Hai Kak @Mr. ZimmermAn, mohon maaf atas ketida...",5.37.1,tolong dong di optimalkan lagi. awal awal saya...,tolong dong di optimalkan lagi awal awal saya ...,"[tolong, dong, di, optimalkan, lagi, awal, awa...","[tolong, dong, di, optimalkan, lagi, awal, awa..."
6,TIDAK SOLUTIF sya order gofood dan pembayaran ...,1,6,5.36.2,"Hai Kak Rossy, mohon maaf atas ketidaknyamanan...",5.36.2,tidak solutif sya order gofood dan pembayaran ...,tidak solutif sya order gofood dan pembayaran ...,"[tidak, solutif, sya, order, gofood, dan, pemb...","[tidak, solutif, sya, order, gofood, dan, pemb..."


In [9]:
# Stopword removal
stop_words = set(stopwords.words('indonesian')) | set(stopwords.words('english'))
stop_words.update(['iya','yaa', 'yg', 'gk','gak','nya','na','sih','ku',"di","ga","ya","gaa","loh","kah","woi","woii","woy", "kyk","dgn","dr","aja","ajaa","klo","kalo","amp","nih", 'aaaaghhhh', 'aae', 'aah',"sih","sihh","tau","terus","toh","tp","tuh","udah","udahh","wkwk","yuk", "zzzzz", 'aaja', 'aaldo', 'aapaaaa'])

reviews['no_stopwords_content'] = reviews['normalized_content'].apply(lambda x: [word for word in x if word not in stop_words])
reviews.head()

,content,score,thumbsUpCount,reviewCreatedVersion,replyContent,appVersion,lowercase_content,cleaned_content,tokenized_content,normalized_content,no_stopwords_content
1,Menu bantuan selalu dijawab oleh chatbot yang ...,1,8,5.36.2,"Hai Kak @supermeind, mohon maaf atas ketidakny...",5.36.2,menu bantuan selalu dijawab oleh chatbot yang ...,menu bantuan selalu dijawab oleh chatbot yang ...,"[menu, bantuan, selalu, dijawab, oleh, chatbot...","[menu, bantuan, selalu, dijawab, oleh, chatbot...","[menu, bantuan, chatbot, menyelesaikan, order,..."
2,"Fitur gofood nya kyk sampah, dikasih opsi peng...",1,108,5.32.1,"Hai Kak Rizi, mohon maaf atas ketidaknyamanann...",5.32.1,"fitur gofood nya kyk sampah, dikasih opsi peng...",fitur gofood nya kyk sampah dikasih opsi penga...,"[fitur, gofood, nya, kyk, sampah, dikasih, ops...","[fitur, gofood, nya, seperti, sampah, dikasih,...","[fitur, gofood, sampah, dikasih, opsi, pengant..."
3,Tolong beri perlindungan untuk nasabah yang pe...,1,53,5.36.2,"Hai Kak Putu, mohon maaf atas ketidaknyamanann...",5.36.2,tolong beri perlindungan untuk nasabah yang pe...,tolong beri perlindungan untuk nasabah yang pe...,"[tolong, beri, perlindungan, untuk, nasabah, y...","[tolong, beri, perlindungan, untuk, nasabah, y...","[tolong, perlindungan, nasabah, pesan, goride,..."
5,tolong dong di optimalkan lagi. awal awal saya...,3,3,5.37.1,"Hai Kak @Mr. ZimmermAn, mohon maaf atas ketida...",5.37.1,tolong dong di optimalkan lagi. awal awal saya...,tolong dong di optimalkan lagi awal awal saya ...,"[tolong, dong, di, optimalkan, lagi, awal, awa...","[tolong, dong, di, optimalkan, lagi, awal, awa...","[tolong, optimalkan, pakai, driver, terdekat, ..."
6,TIDAK SOLUTIF sya order gofood dan pembayaran ...,1,6,5.36.2,"Hai Kak Rossy, mohon maaf atas ketidaknyamanan...",5.36.2,tidak solutif sya order gofood dan pembayaran ...,tidak solutif sya order gofood dan pembayaran ...,"[tidak, solutif, sya, order, gofood, dan, pemb...","[tidak, solutif, sya, order, gofood, dan, pemb...","[solutif, sya, order, gofood, pembayaran, kart..."


In [10]:
# Text Stemming
factory = StemmerFactory()
stemmer = factory.create_stemmer()
reviews['stemmed_content'] = reviews['no_stopwords_content'].apply(lambda x: [stemmer.stem(word) for word in x])
reviews.head()

,content,score,thumbsUpCount,reviewCreatedVersion,replyContent,appVersion,lowercase_content,cleaned_content,tokenized_content,normalized_content,no_stopwords_content,stemmed_content
1,Menu bantuan selalu dijawab oleh chatbot yang ...,1,8,5.36.2,"Hai Kak @supermeind, mohon maaf atas ketidakny...",5.36.2,menu bantuan selalu dijawab oleh chatbot yang ...,menu bantuan selalu dijawab oleh chatbot yang ...,"[menu, bantuan, selalu, dijawab, oleh, chatbot...","[menu, bantuan, selalu, dijawab, oleh, chatbot...","[menu, bantuan, chatbot, menyelesaikan, order,...","[menu, bantu, chatbot, selesai, order, gofood,..."
2,"Fitur gofood nya kyk sampah, dikasih opsi peng...",1,108,5.32.1,"Hai Kak Rizi, mohon maaf atas ketidaknyamanann...",5.32.1,"fitur gofood nya kyk sampah, dikasih opsi peng...",fitur gofood nya kyk sampah dikasih opsi penga...,"[fitur, gofood, nya, kyk, sampah, dikasih, ops...","[fitur, gofood, nya, seperti, sampah, dikasih,...","[fitur, gofood, sampah, dikasih, opsi, pengant...","[fitur, gofood, sampah, kasih, opsi, antar, ce..."
3,Tolong beri perlindungan untuk nasabah yang pe...,1,53,5.36.2,"Hai Kak Putu, mohon maaf atas ketidaknyamanann...",5.36.2,tolong beri perlindungan untuk nasabah yang pe...,tolong beri perlindungan untuk nasabah yang pe...,"[tolong, beri, perlindungan, untuk, nasabah, y...","[tolong, beri, perlindungan, untuk, nasabah, y...","[tolong, perlindungan, nasabah, pesan, goride,...","[tolong, lindung, nasabah, pesan, goride, goca..."
5,tolong dong di optimalkan lagi. awal awal saya...,3,3,5.37.1,"Hai Kak @Mr. ZimmermAn, mohon maaf atas ketida...",5.37.1,tolong dong di optimalkan lagi. awal awal saya...,tolong dong di optimalkan lagi awal awal saya ...,"[tolong, dong, di, optimalkan, lagi, awal, awa...","[tolong, dong, di, optimalkan, lagi, awal, awa...","[tolong, optimalkan, pakai, driver, terdekat, ...","[tolong, optimal, pakai, driver, dekat, cepat,..."
6,TIDAK SOLUTIF sya order gofood dan pembayaran ...,1,6,5.36.2,"Hai Kak Rossy, mohon maaf atas ketidaknyamanan...",5.36.2,tidak solutif sya order gofood dan pembayaran ...,tidak solutif sya order gofood dan pembayaran ...,"[tidak, solutif, sya, order, gofood, dan, pemb...","[tidak, solutif, sya, order, gofood, dan, pemb...","[solutif, sya, order, gofood, pembayaran, kart...","[solutif, sya, order, gofood, bayar, kartu, de..."


In [11]:
# To string after stemming
reviews['final_content'] = reviews['stemmed_content'].apply(lambda x: ' '.join(x))
reviews.head()

,content,score,thumbsUpCount,reviewCreatedVersion,replyContent,appVersion,lowercase_content,cleaned_content,tokenized_content,normalized_content,no_stopwords_content,stemmed_content,final_content
1,Menu bantuan selalu dijawab oleh chatbot yang ...,1,8,5.36.2,"Hai Kak @supermeind, mohon maaf atas ketidakny...",5.36.2,menu bantuan selalu dijawab oleh chatbot yang ...,menu bantuan selalu dijawab oleh chatbot yang ...,"[menu, bantuan, selalu, dijawab, oleh, chatbot...","[menu, bantuan, selalu, dijawab, oleh, chatbot...","[menu, bantuan, chatbot, menyelesaikan, order,...","[menu, bantu, chatbot, selesai, order, gofood,...",menu bantu chatbot selesai order gofood batal ...
2,"Fitur gofood nya kyk sampah, dikasih opsi peng...",1,108,5.32.1,"Hai Kak Rizi, mohon maaf atas ketidaknyamanann...",5.32.1,"fitur gofood nya kyk sampah, dikasih opsi peng...",fitur gofood nya kyk sampah dikasih opsi penga...,"[fitur, gofood, nya, kyk, sampah, dikasih, ops...","[fitur, gofood, nya, seperti, sampah, dikasih,...","[fitur, gofood, sampah, dikasih, opsi, pengant...","[fitur, gofood, sampah, kasih, opsi, antar, ce...",fitur gofood sampah kasih opsi antar cepat sed...
3,Tolong beri perlindungan untuk nasabah yang pe...,1,53,5.36.2,"Hai Kak Putu, mohon maaf atas ketidaknyamanann...",5.36.2,tolong beri perlindungan untuk nasabah yang pe...,tolong beri perlindungan untuk nasabah yang pe...,"[tolong, beri, perlindungan, untuk, nasabah, y...","[tolong, beri, perlindungan, untuk, nasabah, y...","[tolong, perlindungan, nasabah, pesan, goride,...","[tolong, lindung, nasabah, pesan, goride, goca...",tolong lindung nasabah pesan goride gocar rugi...
5,tolong dong di optimalkan lagi. awal awal saya...,3,3,5.37.1,"Hai Kak @Mr. ZimmermAn, mohon maaf atas ketida...",5.37.1,tolong dong di optimalkan lagi. awal awal saya...,tolong dong di optimalkan lagi awal awal saya ...,"[tolong, dong, di, optimalkan, lagi, awal, awa...","[tolong, dong, di, optimalkan, lagi, awal, awa...","[tolong, optimalkan, pakai, driver, terdekat, ...","[tolong, optimal, pakai, driver, dekat, cepat,...",tolong optimal pakai driver dekat cepat kondis...
6,TIDAK SOLUTIF sya order gofood dan pembayaran ...,1,6,5.36.2,"Hai Kak Rossy, mohon maaf atas ketidaknyamanan...",5.36.2,tidak solutif sya order gofood dan pembayaran ...,tidak solutif sya order gofood dan pembayaran ...,"[tidak, solutif, sya, order, gofood, dan, pemb...","[tidak, solutif, sya, order, gofood, dan, pemb...","[solutif, sya, order, gofood, pembayaran, kart...","[solutif, sya, order, gofood, bayar, kartu, de...",solutif sya order gofood bayar kartu debit ord...


## Pelabelan

In [12]:
# Import Lexicon Indonesia Inset
# Lexicon diunduh dari github https://github.com/fajri91/InSet
positive = pd.read_csv('./lexicon_positive.tsv', sep="\t")
negative = pd.read_csv('./lexicon_negative.tsv', sep="\t")
lexicon_pos = dict(zip(positive['word'], positive['weight']))
lexicon_neg = dict(zip(negative['word'], negative['weight']))

In [13]:
def label_sentiment(words):
    score = 0
    for word in words:
        if word in lexicon_pos:
            score += lexicon_pos[word]
        if word in lexicon_neg:
            score += lexicon_neg[word]
    
    label = ""
    if score >= 0:
        label = 'positive'
    elif score < 0:
        label = 'negative'

    return score, label

In [14]:
reviews[['sentiment_score', 'sentiment_label']] = reviews['stemmed_content'].apply(lambda x: pd.Series(label_sentiment(x)))
reviews.head()

,content,score,thumbsUpCount,reviewCreatedVersion,replyContent,appVersion,lowercase_content,cleaned_content,tokenized_content,normalized_content,no_stopwords_content,stemmed_content,final_content,sentiment_score,sentiment_label
1,Menu bantuan selalu dijawab oleh chatbot yang ...,1,8,5.36.2,"Hai Kak @supermeind, mohon maaf atas ketidakny...",5.36.2,menu bantuan selalu dijawab oleh chatbot yang ...,menu bantuan selalu dijawab oleh chatbot yang ...,"[menu, bantuan, selalu, dijawab, oleh, chatbot...","[menu, bantuan, selalu, dijawab, oleh, chatbot...","[menu, bantuan, chatbot, menyelesaikan, order,...","[menu, bantu, chatbot, selesai, order, gofood,...",menu bantu chatbot selesai order gofood batal ...,-20,negative
2,"Fitur gofood nya kyk sampah, dikasih opsi peng...",1,108,5.32.1,"Hai Kak Rizi, mohon maaf atas ketidaknyamanann...",5.32.1,"fitur gofood nya kyk sampah, dikasih opsi peng...",fitur gofood nya kyk sampah dikasih opsi penga...,"[fitur, gofood, nya, kyk, sampah, dikasih, ops...","[fitur, gofood, nya, seperti, sampah, dikasih,...","[fitur, gofood, sampah, dikasih, opsi, pengant...","[fitur, gofood, sampah, kasih, opsi, antar, ce...",fitur gofood sampah kasih opsi antar cepat sed...,0,positive
3,Tolong beri perlindungan untuk nasabah yang pe...,1,53,5.36.2,"Hai Kak Putu, mohon maaf atas ketidaknyamanann...",5.36.2,tolong beri perlindungan untuk nasabah yang pe...,tolong beri perlindungan untuk nasabah yang pe...,"[tolong, beri, perlindungan, untuk, nasabah, y...","[tolong, beri, perlindungan, untuk, nasabah, y...","[tolong, perlindungan, nasabah, pesan, goride,...","[tolong, lindung, nasabah, pesan, goride, goca...",tolong lindung nasabah pesan goride gocar rugi...,-3,negative
5,tolong dong di optimalkan lagi. awal awal saya...,3,3,5.37.1,"Hai Kak @Mr. ZimmermAn, mohon maaf atas ketida...",5.37.1,tolong dong di optimalkan lagi. awal awal saya...,tolong dong di optimalkan lagi awal awal saya ...,"[tolong, dong, di, optimalkan, lagi, awal, awa...","[tolong, dong, di, optimalkan, lagi, awal, awa...","[tolong, optimalkan, pakai, driver, terdekat, ...","[tolong, optimal, pakai, driver, dekat, cepat,...",tolong optimal pakai driver dekat cepat kondis...,-1,negative
6,TIDAK SOLUTIF sya order gofood dan pembayaran ...,1,6,5.36.2,"Hai Kak Rossy, mohon maaf atas ketidaknyamanan...",5.36.2,tidak solutif sya order gofood dan pembayaran ...,tidak solutif sya order gofood dan pembayaran ...,"[tidak, solutif, sya, order, gofood, dan, pemb...","[tidak, solutif, sya, order, gofood, dan, pemb...","[solutif, sya, order, gofood, pembayaran, kart...","[solutif, sya, order, gofood, bayar, kartu, de...",solutif sya order gofood bayar kartu debit ord...,-21,negative


In [15]:
print(reviews['sentiment_label'].value_counts())

sentiment_label
negative    5413
positive    2215
Name: count, dtype: int64


## Data Splitting dan Ekstraksi Fitur

In [16]:
X = reviews['final_content']
y = reviews['sentiment_label']
X_train_text, X_test_text, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

tfidf = TfidfVectorizer()
X_train = tfidf.fit_transform(X_train_text)
X_test = tfidf.transform(X_test_text)

In [17]:
tfidf.get_feature_names_out()

array(['aaplikasi', 'aapnya', 'aaya', ..., 'zona', 'zonasi', 'zoom'],
      shape=(8059,), dtype=object)

## Modeling

In [18]:
model = LinearSVC()
model.fit(X_train, y_train)

,penalty,'l2'
,loss,'squared_hinge'
,dual,'auto'
,tol,0.0001
,C,1.0
,multi_class,'ovr'
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,verbose,0
,random_state,None


In [19]:
# Prediksi sentimen pada data pelatihan dan data uji
y_pred_train_nb = model.predict(X_train.toarray())
y_pred_test_nb = model.predict(X_test.toarray())


In [20]:
print("Classification Report train dataset")
print(classification_report(y_train, y_pred_train_nb))

print("Classification Report test dataset")
print(classification_report(y_test, y_pred_test_nb))

Classification Report train dataset
              precision    recall  f1-score   support

    negative       0.99      1.00      1.00      4337
    positive       1.00      0.98      0.99      1765

    accuracy                           0.99      6102
   macro avg       0.99      0.99      0.99      6102
weighted avg       0.99      0.99      0.99      6102

Classification Report test dataset
              precision    recall  f1-score   support

    negative       0.90      0.95      0.92      1076
    positive       0.85      0.74      0.79       450

    accuracy                           0.88      1526
   macro avg       0.87      0.84      0.86      1526
weighted avg       0.88      0.88      0.88      1526



In [21]:
print("Confusion Matrix test dataset")
print(confusion_matrix(y_test, y_pred_test_nb))

Confusion Matrix test dataset
[[1019   57]
 [ 119  331]]
